In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    roc_curve, confusion_matrix,
    precision_recall_curve
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score


In [2]:
df = pd.read_csv('final_df.csv')

drop_cols = [
    'permalink',        # identifier
    'homepage_url',     # identifier
    'status',           # leakage — encodes the target directly
    'funding_total_usd' # replaced by log_funding
]

redundant_cols = [
    'founded_at',       # captured by founded_year + startup_age_days
    'founded_month',    # noise — year is sufficient
    'founded_quarter',  # noise — year is sufficient
    'first_funding_at', # captured by startup_age_days
    'last_funding_at',  # captured by funding_duration_days
]

df = df.drop(columns=drop_cols + redundant_cols)
print(f"Dataset shape after dropping columns: {df.shape}")

Dataset shape after dropping columns: (48121, 34)


In [3]:
y = df['success'].copy()
X = df.drop(columns=['name', 'success']).copy()

# Extract primary category from pipe-delimited string
X['primary_category'] = (
    X['category_list']
    .str.strip('|')
    .str.split('|')
    .str[0]
    .str.strip()
    .fillna('Unknown')
)
X = X.drop(columns=['category_list'])

# Drop high-cardinality geo columns
X = X.drop(columns=['region', 'city'], errors='ignore')

In [4]:
round_cols = ['seed', 'venture', 'round_A', 'round_B', 'round_C',
              'round_D', 'round_E', 'round_F', 'round_G', 'round_H']

round_order = {
    'seed': 1, 'venture': 2, 'round_A': 3, 'round_B': 4, 'round_C': 5,
    'round_D': 6, 'round_E': 7, 'round_F': 8, 'round_G': 9, 'round_H': 10
}

# Highest funding round reached (ordinal signal)
X['highest_round'] = X[round_cols].apply(
    lambda row: max([round_order[c] for c in round_cols if row[c] > 0], default=0),
    axis=1
)

# Number of distinct round types received
X['round_diversity'] = (X[round_cols] > 0).sum(axis=1)

# Funding intensity features
X['funding_per_round'] = np.expm1(X['log_funding']) / (X['funding_rounds'] + 1)
X['funding_per_year']  = np.expm1(X['log_funding']) / (X['startup_age_days'] / 365 + 1)
X['rounds_per_year']   = X['funding_rounds'] / (X['funding_duration_days'] / 365 + 1)

# Drop sparse round-type columns (>95% zeros)
sparse_cols = [
    'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing',
    'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt',
    'secondary_market', 'product_crowdfunding'
]
for col in sparse_cols:
    if col in X.columns and (X[col] == 0).mean() > 0.95:
        X = X.drop(columns=[col])
print(f"Engineered features added: highest_round, round_diversity, "
      f"funding_per_round, funding_per_year, rounds_per_year")

Engineered features added: highest_round, round_diversity, funding_per_round, funding_per_year, rounds_per_year


In [5]:
cat_cols = ['primary_category', 'market', 'country_code', 'state_code']

# Fix unknown state codes
X['state_code'] = X['state_code'].replace('Unknown', np.nan).fillna('Unknown')

# Cast to category dtype (used by LightGBM natively)
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

# Sanity check
print(f"Final X shape: {X.shape}")
print(f"Any NaNs: {X.isnull().sum().sum()}")
print(f"Any Inf:  {np.isinf(X.select_dtypes('number')).sum().sum()}")
print(f"\nDtype summary:\n{X.dtypes.value_counts()}")

Final X shape: (48121, 26)
Any NaNs: 0
Any Inf:  63

Dtype summary:
float64     20
int64        2
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [6]:
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X.copy()
X_encoded[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))

# Replace any inf values produced by feature engineering edge cases
X_encoded = X_encoded.replace([np.inf, -np.inf], np.nan)
X_encoded = X_encoded.fillna(X_encoded.median(numeric_only=True))

assert not np.isinf(X_encoded.select_dtypes('number').values).any(), "Inf values remain"
assert not X_encoded.isnull().any().any(), "NaN values remain"
print(f"Encoded feature matrix ready: {X_encoded.shape}")

Encoded feature matrix ready: (48121, 26)


In [7]:
# Tiny dataset — 500 rows, no sklearn dependencies
X_np = X_encoded.values[:500].astype('float32')
y_np = y.values[:500].astype('float32')

X_t = torch.tensor(X_np)
y_t = torch.tensor(y_np)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=128, shuffle=True)

model = nn.Sequential(
    nn.Linear(X_np.shape[1], 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid()
)
opt  = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

for epoch in range(5):
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(xb).squeeze(), yb).backward()
        opt.step()
    print(f'Epoch {epoch+1} done')

print('Finished')

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Finished


In [ ]:
# Sample and convert to float32 immediately to save memory
X_np = X_encoded.sample(2000, random_state=42).astype('float32').values
y_np = y.loc[X_encoded.sample(2000, random_state=42).index].values

# Free the large encoded matrix from memory
del X_encoded
import gc; gc.collect()

# Split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_np, y_np, test_size=0.2, stratify=y_np, random_state=42
)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr).astype('float32')
X_val  = scaler.transform(X_val).astype('float32')

# Tensors
X_tr_t = torch.tensor(X_tr)
y_tr_t = torch.tensor(y_tr.astype('float32'))
X_val_t = torch.tensor(X_val)

loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                    batch_size=256, shuffle=True)

# Model
model   = nn.Sequential(
    nn.Linear(X_tr.shape[1], 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),  nn.Sigmoid()
)
opt     = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCELoss()

# Train
for epoch in range(10):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(xb).squeeze(), yb).backward()
        opt.step()
    print(f'Epoch {epoch+1}/10 done')

# Evaluate
model.eval()
with torch.no_grad():
    proba = model(X_val_t).squeeze().numpy()

preds = (proba >= 0.5).astype(int)
nn_scores = {
    'roc_auc':  roc_auc_score(y_val, proba),
    'pr_auc':   average_precision_score(y_val, proba),
    'f1_macro': f1_score(y_val, preds, average='macro'),
    'proba':    proba,
    'true':     y_val,
}

print(f"\n── Neural Network Results ──")
print(f"ROC-AUC:  {nn_scores['roc_auc']:.4f}")
print(f"PR-AUC:   {nn_scores['pr_auc']:.4f}")
print(f"F1-Macro: {nn_scores['f1_macro']:.4f}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Stratified subsample (preserves class ratio) ──────────────────────────────
X_small, _, y_small, _ = train_test_split(
    X_encoded, y,
    train_size=2000,
    stratify=y,
    random_state=42
)
print(f"Subsample shape: {X_small.shape}")
print(f"Class distribution:\n{y_small.value_counts()}")
print(f"Positive rate: {y_small.mean():.2%}")

# ── Architecture ──────────────────────────────────────────────────────────────
class StartupNet(nn.Module):
    """
    Fully connected feedforward neural network for binary classification.
    - 2 hidden layers with ReLU activations
    - Dropout for regularization
    - Sigmoid output for probability estimation
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            # Hidden layer 1
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Hidden layer 2
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output layer — sigmoid for binary classification
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ── Training loop ─────────────────────────────────────────────────────────────
def train_neural_net(X_tr, y_tr, X_val, y_val, epochs=10, batch_size=512, lr=1e-3):
    # Standardize features (required for neural nets)
    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)

    # Convert to tensors
    X_tr_t  = torch.tensor(X_tr_s,      dtype=torch.float32)
    y_tr_t  = torch.tensor(y_tr.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val_s,     dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                        batch_size=batch_size, shuffle=True)

    model     = StartupNet(X_tr_s.shape[1])
    criterion = nn.BCELoss()

    # Adam optimizer with weight decay (L2 regularization)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Gradient descent over epochs
    best_val_auc, best_proba = 0, None
    for epoch in range(epochs):
        # ── Training phase ──
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()    # backpropagation
            optimizer.step()   # gradient descent step
            epoch_loss += loss.item()

        # ── Validation phase ──
        model.eval()
        with torch.no_grad():
            proba = model(X_val_t).numpy()

        val_auc = roc_auc_score(y_val, proba)

        # Save best checkpoint based on validation AUC
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_proba   = proba.copy()

        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:>3}/{epochs} | '
                  f'Loss: {epoch_loss/len(loader):.4f} | '
                  f'Val AUC: {val_auc:.4f}')

    return best_proba


# Single 80/20 split — no CV loop
X_tr, X_val, y_tr, y_val = train_test_split(
    X_small.values, y_small,
    test_size=0.2, stratify=y_small, random_state=42
)

proba = train_neural_net(X_tr, y_tr, X_val, y_val,
                         epochs=10, batch_size=512, lr=1e-3)
preds = (proba >= 0.5).astype(int)

nn_scores = {
    'roc_auc':  roc_auc_score(y_val, proba),
    'pr_auc':   average_precision_score(y_val, proba),
    'f1_macro': f1_score(y_val, preds, average='macro'),
    'proba':    np.array(proba),
    'true':     np.array(y_val),
}

print(f"── Neural Network (MLP) Results ──")
print(f"ROC-AUC:  {nn_scores['roc_auc']:.4f}")
print(f"PR-AUC:   {nn_scores['pr_auc']:.4f}")
print(f"F1-Macro: {nn_scores['f1_macro']:.4f}")

nn_scores = {
    'roc_auc':  np.mean(nn_roc),
    'pr_auc':   np.mean(nn_pr),
    'f1_macro': np.mean(nn_f1),
    'proba':    np.array(nn_proba_all),
    'true':     np.array(nn_true_all),
}

print(f'\n── Neural Network (MLP) Results ──')
print(f"ROC-AUC:  {nn_scores['roc_auc']:.4f} ± {np.std(nn_roc):.4f}")
print(f"PR-AUC:   {nn_scores['pr_auc']:.4f} ± {np.std(nn_pr):.4f}")
print(f"F1-Macro: {nn_scores['f1_macro']:.4f} ± {np.std(nn_f1):.4f}")

In [ ]:
all_scores = {
    'Neural Network': nn_scores,
}

# ── Print summary table ───────────────────────────────────────────────────────
print("\n── Model Comparison ──")
print(f"{'Model':<20} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1-Macro':>10}")
print("─" * 52)
for name, s in all_scores.items():
    print(f"{name:<20} {s['roc_auc']:>10.4f} {s['pr_auc']:>10.4f} {s['f1_macro']:>10.4f}")

# ── Plot setup ────────────────────────────────────────────────────────────────
colors      = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
model_names = list(all_scores.keys())
fig         = plt.figure(figsize=(18, 14))
gs          = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

# ── 1. ROC Curves ─────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for (name, s), color in zip(all_scores.items(), colors):
    fpr, tpr, _ = roc_curve(s['true'], s['proba'])
    ax1.plot(fpr, tpr, lw=2, color=color,
             label=f"{name}  (AUC = {s['roc_auc']:.4f})")
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.01])

# ── 2. Precision-Recall Curves ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
baseline = y.mean()
for (name, s), color in zip(all_scores.items(), colors):
    prec, rec, _ = precision_recall_curve(s['true'], s['proba'])
    ax2.plot(rec, prec, lw=2, color=color,
             label=f"{name}  (PR-AUC = {s['pr_auc']:.4f})")
ax2.axhline(baseline, color='k', linestyle='--', lw=1,
            label=f'Random baseline ({baseline:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.01])

# ── 3. Confusion Matrices ─────────────────────────────────────────────────────
inner_gs = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[1, 0],
                                             hspace=0.5, wspace=0.4)
for i, (name, s) in enumerate(all_scores.items()):
    ax = fig.add_subplot(inner_gs[i // 2, i % 2])
    preds = (s['proba'] >= 0.5).astype(int)
    cm    = confusion_matrix(s['true'], preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, linewidths=0.5,
                xticklabels=['Fail', 'Success'],
                yticklabels=['Fail', 'Success'])
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual', fontsize=8)
    ax.tick_params(labelsize=7)

# ── 4. Metric Bar Chart ───────────────────────────────────────────────────────
ax4      = fig.add_subplot(gs[1, 1])
metrics  = ['roc_auc', 'pr_auc', 'f1_macro']
labels   = ['ROC-AUC', 'PR-AUC', 'F1-Macro']
x        = np.arange(len(labels))
width    = 0.18

for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [all_scores[name][m] for m in metrics]
    bars = ax4.bar(x + i * width, vals, width, label=name,
                   color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)

ax4.set_xticks(x + width * 1.5)
ax4.set_xticklabels(labels, fontsize=11)
ax4.set_ylim(0, 1.05)
ax4.set_ylabel('Score')
ax4.set_title('Metric Comparison', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8)

fig.suptitle('Model Comparison — Out-of-Fold Predictions',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()